# ДЗ №4. Задание 2
### Выполнил: Лёвин В.А.
### Группа: ИУ6-54Б

Коллаборативная фильтрация.


## Подготовка
Задача № 2:  1

In [8]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

## Малый набор MovieLens

### 1. Разделите данные с рейтингами на обучающее (train_init - 0.8) и тестовое подмножества (test - 0.2), определите среднее значение рейтинга в обучающем подмножестве и вычислите rmse для тестового подмножества, если для всех значений из test предсказывается среднее значение рейтинга

In [9]:
def load_split(folder):
    types = {"userId": "int32", "movieId": "int32", "rating": "float32"}
    ratings = pd.read_csv(f"{folder}/ratings.csv", usecols=types.keys(), dtype=types)
    movies = pd.read_csv(f"{folder}/movies.csv", usecols=["movieId", "genres"])
    rng = np.random.default_rng(42)
    train_mask = rng.random(len(ratings), dtype=np.float32) < 0.8
    train = ratings[train_mask]
    test = ratings[~train_mask]
    shape = (int(ratings.userId.max()) + 1, int(movies.movieId.max()) + 1)
    train_init = csr_matrix((train.rating, (train.userId, train.movieId)), shape=shape)
    return train_init, test, movies


def rmse(actual, predicted):
    return np.sqrt(np.mean((actual - predicted) ** 2, dtype=np.float64))


def show_baseline(train_init, test):
    mean = train_init.data.mean(dtype=np.float64)
    print(f"Средняя оценка train_init: {mean:.4f}")
    print(f"RMSE среднего на test: {rmse(test.rating.to_numpy(), mean):.4f}")


train_init, test, movies = load_split("../ml-latest-small")
show_baseline(train_init, test)

Средняя оценка train_init: 3.5021
RMSE среднего на test: 1.0500


### 2. Реализуйте коллаборативную фильтрацию в соответствии с вариантом. Для определения схожести используйте train_init, для расчета `rmse` - test

Профиль пользователя показывает, сколько баллов он поставил фильмам каждого жанра. После нормировки скалярное произведение двух профилей даёт косинусное сходство пользователей.

Прогноз начинаем со средней оценки пользователя. К ней прибавляем поправку от тех, кто оценил фильм. Например, если сосед обычно ставит 3, а фильму поставил 5, его вклад в поправку равен +2 с весом, равным сходству. Чтобы на полном наборе не перебирать соседей для каждого прогноза, заранее считаем две суммы по каждому фильму.

In [15]:
def user_profiles(train_init, movies):
    genres = movies.genres.str.get_dummies(sep="|")
    movie_genres = np.zeros((train_init.shape[1], genres.shape[1]))
    movie_genres[movies.movieId] = genres

    profiles = train_init @ movie_genres
    lengths = np.linalg.norm(profiles, axis=1, keepdims=True)
    lengths[lengths == 0] = 1
    return profiles / lengths


def movie_totals(train_init, profiles, user_means, counts):
    votes = train_init.copy()
    # Оценки в разреженной матрице записаны по пользователям.
    votes.data -= np.repeat(user_means, counts)
    deviation_sums = votes.T @ profiles

    votes.data.fill(1)
    profile_sums = votes.T @ profiles
    return deviation_sums, profile_sums


def predict_ratings(train_init, test, movies):
    profiles = user_profiles(train_init, movies)
    counts = train_init.getnnz(axis=1)
    user_means = train_init.sum(axis=1).A1 / np.maximum(counts, 1)
    user_means[counts == 0] = train_init.data.mean(dtype=np.float64)
    deviation_sums, profile_sums = movie_totals(train_init, profiles, user_means, counts)

    predictions = np.empty(len(test))
    batch_size = 50_000
    for start in range(0, len(test), batch_size):
        batch = test.iloc[start:start + batch_size]
        users = profiles[batch.userId]
        weighted_sum = (users * deviation_sums[batch.movieId]).sum(axis=1)
        weight_sum = (users * profile_sums[batch.movieId]).sum(axis=1)
        adjustment = np.zeros(len(batch))
        np.divide(weighted_sum, weight_sum, out=adjustment, where=weight_sum > 0)
        predictions[start:start + len(batch)] = user_means[batch.userId] + adjustment
    return np.clip(predictions, 0.5, 5)


predictions = predict_ratings(train_init, test, movies)
predictions 

array([3.88351277, 4.35528162, 4.26752625, ..., 2.80718056, 2.13800938,
       2.73973586])

### 3. Определите rmse для тестового подмножества

Учитываем каждую строку test. Если сумма сходств нулевая, используем среднюю оценку пользователя. Для нового пользователя берём общее среднее по train_init.

In [11]:
print(f"RMSE по схожести пользователей: {rmse(test.rating.to_numpy(), predictions):.4f}")

RMSE по схожести пользователей: 0.9000


## Полный набор MovieLens

### 1. Разделите данные с рейтингами на обучающее (train_init - 0.8) и тестовое подмножества (test - 0.2), определите среднее значение рейтинга в обучающем подмножестве и вычислите rmse для тестового подмножества, если для всех значений из test предсказывается среднее значение рейтинга

In [12]:
train_init, test, movies = load_split("../ml-latest")
show_baseline(train_init, test)

Средняя оценка train_init: 3.5427
RMSE среднего на test: 1.0638


### 2. Реализуйте коллаборативную фильтрацию в соответствии с вариантом. Для определения схожести используйте train_init, для расчета `rmse` - test

In [13]:
predictions = predict_ratings(train_init, test, movies)

### 3. Определите rmse для тестового подмножества

In [14]:
print(f"RMSE по схожести пользователей: {rmse(test.rating.to_numpy(), predictions):.4f}")

RMSE по схожести пользователей: 0.8772
